In [1]:
print("j")

j


In [2]:
# Search tool for ItemMaster.xlsx
# This code defines a reusable `search_items(query, top=10)` function you can call in this notebook.
# It uses token overlap + SequenceMatcher ratio to score matches. No external libraries required.
# The function returns a dataframe of top matches and also displays it in an interactive table.
# Example usage shown at the end (search for your SUPRATHERME query).

import pandas as pd
import re
from difflib import SequenceMatcher
from ace_tools import display_dataframe_to_user

# Load the file
df = pd.read_excel('/mnt/data/ItemMaster.xlsx')

# Basic cleanup and a searchable column
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).upper().strip()
    s = re.sub(r'[^A-Z0-9.\- ]+', ' ', s)  # keep alnum, dot, dash, space
    s = re.sub(r'\s+', ' ', s)
    return s

df['__desc_search'] = df['Description'].apply(clean_text).fillna('')
df['__code_search'] = df['Item Code'].astype(str).str.upper()

def token_set(text):
    return set([t for t in re.split(r'[\s\-xX]+', text) if t])

def similarity_score(query, text):
    # Token overlap score (Jaccard)
    qtok = token_set(query)
    ttok = token_set(text)
    if len(qtok) == 0 and len(ttok) == 0:
        token_score = 0.0
    else:
        inter = qtok.intersection(ttok)
        union = qtok.union(ttok)
        token_score = len(inter) / len(union)
    # Sequence ratio (character-level)
    seq_ratio = SequenceMatcher(None, query, text).ratio()
    # Combined score (weights can be adjusted)
    score = 0.65 * token_score + 0.35 * seq_ratio
    return score

def search_items(query, top=10, show=True):
    q = clean_text(query)
    results = []
    for _, row in df.iterrows():
        txt = row['__desc_search']
        sc = similarity_score(q, txt)
        results.append((sc, row['Item Code'], row['Description'], row.get('Location', ''), row.get('Pack Size', ''), row.get('Type', ''), row.get('HS Code', ''), row.get('Date updated', '')))
    res_df = pd.DataFrame(results, columns=['score','Item Code','Description','Location','Pack Size','Type','HS Code','Date updated'])
    res_df = res_df.sort_values('score', ascending=False).reset_index(drop=True)
    if show:
        display_dataframe_to_user("Search results for: " + query, res_df.head(top))
    return res_df.head(top)

# Demonstration search (your example)
demo = search_items("SUPRATHERME VP 3.15X450 - 2KGS", top=10)
demo


ModuleNotFoundError: No module named 'ace_tools'